### NYC DOT Marketing Mix Modeling -- Preprocessing

#### Data Sources

I obtained data from two publicly available NYC Open Data datasets:

- **Advertising expenditures:** [Local Law 83 – City Agency Advertising Spend](https://data.cityofnewyork.us/City-Government/Local-Law-83-City-Agency-Advertising-Spend/n56u-3hzv/about_data)
- **Motor vehicle crashes:** [Motor Vehicle Collisions – Crashes](https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Crashes/h9gi-nx95/about_data)

The advertising data was downloaded as a CSV file containing expenditures across NYC agencies and advertising campaigns. I focused on NYC Department of Transportation (DOT) road-safety advertising and citywide motor vehicle crashes to investigate whether advertising expenditures were associated with changes in crash frequency.

#### Advertising Expenditure Processing

The original advertising dataset contained individual expenditures associated with campaign run dates rather than weekly spending totals. I processed these records to construct a weekly advertising dataset suitable for marketing mix modeling (MMM).

The processing involved:

1. Filtering advertising records to NYC DOT.
2. Identifying road-safety campaigns using keywords including Vision Zero, Speed Camera, Truck Safety, DWI, Work Zone, and E-Bike Safety.
3. Standardizing expenditures into seven media categories: TV, Radio, Print, Digital, Social, Out-of-Home (OOH), and Other.
4. Addressing date inconsistencies, including reversed start/end dates, missing end dates, and records containing multiple insertion dates.
5. Allocating advertising expenditures evenly across each campaign's scheduled run dates and aggregating spending into Monday-start weeks.
6. Separating road-safety advertising expenditures from other NYC DOT advertising spending.

Because actual daily spending and advertising delivery were unavailable, I assumed expenditures were distributed uniformly throughout each advertising buy's scheduled run dates.

#### Crash Data Integration

I aggregated daily NYC motor vehicle collision records into weekly totals for crashes, persons injured, and persons killed.

I then joined these outcomes to the weekly advertising expenditure dataset using `week_start`, ensuring both datasets followed the same Monday-start weekly structure.

Additional variables were created for the ISO week number, calendar month, sequential time index, and number of observed crash days per week.

#### Study Period and Outcome Selection

I selected fiscal years 2023–2025 (July 1, 2022, through June 30, 2025) to obtain approximately three years of historical observations.

I selected **weekly motor vehicle crashes as the primary modeling outcome** because crashes can occur without resulting in injuries or fatalities. Crash counts therefore provide a broader measure of collision frequency, whereas injuries and fatalities represent more severe outcomes.

The resulting dataset contains **158 weekly observations**, spanning June 27, 2022, through June 30, 2025. Although all 158 weeks contain seven days of crash observations, the first and final weeks have incomplete advertising expenditure coverage due to the selected spending window. These observations require consideration during preprocessing.

#### Data Limitations

The advertising dataset contains expenditures rather than actual advertising exposure metrics such as impressions, reach, or frequency. This makes it more difficult to evaluate the relationship between advertising activity and crash outcomes because spending does not necessarily reflect how many people were exposed to a campaign.

Additionally, allocating spending evenly across campaign run dates introduces an assumption about advertising activity that may not reflect actual daily delivery patterns.

Finally, the datasets are observational. Factors unrelated to advertising may influence crash frequency, and advertising spending may itself respond to changes in road-safety conditions. Consequently, modeled relationships should not automatically be interpreted as causal effects.

#### Import libraries

In [1]:
# Core analysis libraries
import pandas as pd
import numpy as np
from datetime import date, timedelta
from pathlib import Path

# Show all columns
pd.set_option('display.max_columns', None)

#### File paths

In [2]:
# File paths
PROJECT_ROOT = Path.cwd().resolve().parent
SRC_DIR = PROJECT_ROOT / "src"
CLEAN_DATA_DIR = PROJECT_ROOT / "data/clean_data"
RAW_DATA_DIR = PROJECT_ROOT / "data/raw_data"
MODELS_DIR = PROJECT_ROOT / "models"
DEPLOYMENT_DIR = PROJECT_ROOT / "deployment"

#### Import data

In [3]:
data = pd.read_csv(RAW_DATA_DIR / "dot_mmm_panel.csv")

data.head()

,week_start,spend_safety_tv,spend_safety_radio,spend_safety_print,spend_safety_digital,spend_safety_social,spend_safety_ooh,spend_safety_other,spend_safety_total,spend_nonsafety_total,spend_total,crashes,injured,killed,days_observed,partial_week,iso_week,month,t
0,2022-06-27,0.00,0.00,0.00,0.0,0.0,0.0,0.00,0.00,1814.00,1814.00,2066,1116,7,7,0,26,6,0
1,2022-07-04,0.00,0.00,0.00,0.0,0.0,0.0,0.00,0.00,274.00,274.00,1919,1053,12,7,0,27,7,1
2,2022-07-11,21920.83,3129.63,3209.92,0.0,0.0,0.0,18135.30,46395.68,9356.00,55751.68,1994,1191,9,7,0,28,7,2
3,2022-07-18,25574.30,3651.24,3744.90,0.0,0.0,0.0,21157.85,54128.29,41290.56,95418.85,2086,1183,10,7,0,29,7,3
4,2022-07-25,25574.30,3651.24,53479.90,5730.0,0.0,0.0,35958.85,124394.29,68711.50,193105.79,2044,1020,5,7,0,30,7,4


In [4]:
data.shape

(158, 19)

In [5]:
# Standardize column names
data.columns = (
    data.columns.str.strip()
             .str.lower()
             .str.replace(" ", "_")
             .str.replace("-", "_")
)

In [6]:
data.columns

Index(['week_start', 'spend_safety_tv', 'spend_safety_radio',
       'spend_safety_print', 'spend_safety_digital', 'spend_safety_social',
       'spend_safety_ooh', 'spend_safety_other', 'spend_safety_total',
       'spend_nonsafety_total', 'spend_total', 'crashes', 'injured', 'killed',
       'days_observed', 'partial_week', 'iso_week', 'month', 't'],
      dtype='object')

#### Data types and nulls

In [7]:
# Data types and nulls
data_types_null_counts = pd.DataFrame()
data_types_null_counts["Data Type"] = data.dtypes
data_types_null_counts["Count of Nulls"] = data.isna().sum()
data_types_null_counts

,Data Type,Count of Nulls
week_start,object,0
spend_safety_tv,float64,0
spend_safety_radio,float64,0
spend_safety_print,float64,0
spend_safety_digital,float64,0
spend_safety_social,float64,0
spend_safety_ooh,float64,0
spend_safety_other,float64,0
spend_safety_total,float64,0
spend_nonsafety_total,float64,0


In [8]:
data.describe().round(3)

,spend_safety_tv,spend_safety_radio,spend_safety_print,spend_safety_digital,spend_safety_social,spend_safety_ooh,spend_safety_other,spend_safety_total,spend_nonsafety_total,spend_total,crashes,injured,killed,days_observed,partial_week,iso_week,month,t
count,158.000,158.000,158.000,158.000,158.000,158.000,158.000,158.000,158.000,158.000,158.000,158.000,158.000,158.0,158.0,158.000,158.000,158.000
mean,17461.797,16622.804,5862.203,4966.393,2687.203,4575.962,21435.735,73612.095,10867.105,84479.200,1799.671,1016.310,5.241,7.0,0.0,26.500,6.487,78.500
std,33231.002,32921.774,11179.996,9322.968,8871.314,17862.186,38534.060,125133.905,27102.679,126359.127,168.440,111.207,2.626,0.0,0.0,14.961,3.443,45.755
min,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1374.000,723.000,0.000,7.0,0.0,1.000,1.000,0.000
25%,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1697.000,946.750,3.000,7.0,0.0,14.000,4.000,39.250
50%,0.000,0.000,0.000,0.000,0.000,0.000,0.000,94.955,78.125,23850.685,1803.500,1023.000,5.000,7.0,0.0,26.500,6.500,78.500
75%,25574.300,10614.040,4250.000,3629.192,0.000,0.000,22812.500,64738.305,7899.575,107459.052,1916.500,1109.000,7.000,7.0,0.0,39.000,9.000,117.750
max,200239.520,109943.770,60952.900,37968.470,41702.130,102147.930,159086.470,468792.840,141937.730,493203.260,2188.000,1267.000,12.000,7.0,0.0,52.000,12.000,157.000


#### Convert `week_start` to datetime

In [9]:
data["week_start"] = pd.to_datetime(data["week_start"])
data["week_start"].dtypes

dtype('<M8[ns]')

#### Holiday Indicator

Binary indicator to identify weeks containing US holidays, allowing the model to account for potential changes in crash frequency during holiday periods.

In [10]:
# Import library
import holidays

In [11]:
# Create dictionary of US holidays
us_holidays = holidays.US(years=[2022, 2023, 2024, 2025])

In [12]:
# Add column to indicate if the week has a holiday
data["is_holiday_week"] = data["week_start"].apply(
    lambda week: int(
        any(
            (week + pd.Timedelta(days=i)).date() in us_holidays
            for i in range(7)
        )
    )
)

data.head()

,week_start,spend_safety_tv,spend_safety_radio,spend_safety_print,spend_safety_digital,spend_safety_social,spend_safety_ooh,spend_safety_other,spend_safety_total,spend_nonsafety_total,spend_total,crashes,injured,killed,days_observed,partial_week,iso_week,month,t,is_holiday_week
0,2022-06-27,0.00,0.00,0.00,0.0,0.0,0.0,0.00,0.00,1814.00,1814.00,2066,1116,7,7,0,26,6,0,0
1,2022-07-04,0.00,0.00,0.00,0.0,0.0,0.0,0.00,0.00,274.00,274.00,1919,1053,12,7,0,27,7,1,1
2,2022-07-11,21920.83,3129.63,3209.92,0.0,0.0,0.0,18135.30,46395.68,9356.00,55751.68,1994,1191,9,7,0,28,7,2,0
3,2022-07-18,25574.30,3651.24,3744.90,0.0,0.0,0.0,21157.85,54128.29,41290.56,95418.85,2086,1183,10,7,0,29,7,3,0
4,2022-07-25,25574.30,3651.24,53479.90,5730.0,0.0,0.0,35958.85,124394.29,68711.50,193105.79,2044,1020,5,7,0,30,7,4,0


In [13]:
# Check counts
data["is_holiday_week"].value_counts()

is_holiday_week
0    123
1     35
Name: count, dtype: int64

#### Congestion Pricing Indicator

Binary indicator for weeks following the introduction of NYC congestion pricing in January 2025 to account for a policy change that could influence crash frequency.

In [14]:
# Congestion charge
data["post_congestion"] = (data["week_start"] >= "2025-01-06").astype(int)
data.head()

,week_start,spend_safety_tv,spend_safety_radio,spend_safety_print,spend_safety_digital,spend_safety_social,spend_safety_ooh,spend_safety_other,spend_safety_total,spend_nonsafety_total,spend_total,crashes,injured,killed,days_observed,partial_week,iso_week,month,t,is_holiday_week,post_congestion
0,2022-06-27,0.00,0.00,0.00,0.0,0.0,0.0,0.00,0.00,1814.00,1814.00,2066,1116,7,7,0,26,6,0,0,0
1,2022-07-04,0.00,0.00,0.00,0.0,0.0,0.0,0.00,0.00,274.00,274.00,1919,1053,12,7,0,27,7,1,1,0
2,2022-07-11,21920.83,3129.63,3209.92,0.0,0.0,0.0,18135.30,46395.68,9356.00,55751.68,1994,1191,9,7,0,28,7,2,0,0
3,2022-07-18,25574.30,3651.24,3744.90,0.0,0.0,0.0,21157.85,54128.29,41290.56,95418.85,2086,1183,10,7,0,29,7,3,0,0
4,2022-07-25,25574.30,3651.24,53479.90,5730.0,0.0,0.0,35958.85,124394.29,68711.50,193105.79,2044,1020,5,7,0,30,7,4,0,0


In [15]:
# Check counts
data["post_congestion"].value_counts()

post_congestion
0    132
1     26
Name: count, dtype: int64

#### Check date range

In [16]:
print("Earliest date", min(data["week_start"]))
print("Most recent date", max(data["week_start"]))

Earliest date 2022-06-27 00:00:00
Most recent date 2025-06-30 00:00:00


#### Sort by date confirmation

In [17]:
data = data.sort_values(by="week_start", ascending=True).copy()
data.head(10)

,week_start,spend_safety_tv,spend_safety_radio,spend_safety_print,spend_safety_digital,spend_safety_social,spend_safety_ooh,spend_safety_other,spend_safety_total,spend_nonsafety_total,spend_total,crashes,injured,killed,days_observed,partial_week,iso_week,month,t,is_holiday_week,post_congestion
0,2022-06-27,0.00,0.00,0.00,0.00,0.0,0.0,0.00,0.00,1814.00,1814.00,2066,1116,7,7,0,26,6,0,0,0
1,2022-07-04,0.00,0.00,0.00,0.00,0.0,0.0,0.00,0.00,274.00,274.00,1919,1053,12,7,0,27,7,1,1,0
2,2022-07-11,21920.83,3129.63,3209.92,0.00,0.0,0.0,18135.30,46395.68,9356.00,55751.68,1994,1191,9,7,0,28,7,2,0,0
3,2022-07-18,25574.30,3651.24,3744.90,0.00,0.0,0.0,21157.85,54128.29,41290.56,95418.85,2086,1183,10,7,0,29,7,3,0,0
4,2022-07-25,25574.30,3651.24,53479.90,5730.00,0.0,0.0,35958.85,124394.29,68711.50,193105.79,2044,1020,5,7,0,30,7,4,0,0
5,2022-08-01,25574.30,3651.24,60952.90,2996.45,0.0,0.0,36106.01,129280.90,37677.68,166958.58,2113,1127,5,7,0,31,8,5,0,0
6,2022-08-08,25574.30,3651.24,19360.90,2996.45,0.0,0.0,36106.01,87688.90,605.21,88294.11,2114,1159,7,7,0,32,8,6,0,0
7,2022-08-15,25574.30,3651.24,12705.90,2996.45,0.0,0.0,36106.01,81033.90,605.21,81639.11,1944,1115,4,7,0,33,8,7,0,0
8,2022-08-22,25574.30,3651.24,25400.90,2996.45,0.0,0.0,36106.01,93728.90,245.04,93973.94,1917,994,8,7,0,34,8,8,0,0
9,2022-08-29,25574.30,3651.24,5805.90,1284.19,0.0,0.0,27564.20,63879.84,65.81,63945.64,1976,1112,7,7,0,35,8,9,0,0


#### Remove incomplete spending weeks

In [18]:
data = data[
    ~data["week_start"].isin(
        pd.to_datetime(["2022-06-27", "2025-06-30"])
    )
].copy()

data.shape

(156, 21)

#### Reindex `t` column

In [19]:
data["t"] = range(len(data))
data = data.reset_index(drop=True)
data.head(5)

,week_start,spend_safety_tv,spend_safety_radio,spend_safety_print,spend_safety_digital,spend_safety_social,spend_safety_ooh,spend_safety_other,spend_safety_total,spend_nonsafety_total,spend_total,crashes,injured,killed,days_observed,partial_week,iso_week,month,t,is_holiday_week,post_congestion
0,2022-07-04,0.00,0.00,0.00,0.00,0.0,0.0,0.00,0.00,274.00,274.00,1919,1053,12,7,0,27,7,0,1,0
1,2022-07-11,21920.83,3129.63,3209.92,0.00,0.0,0.0,18135.30,46395.68,9356.00,55751.68,1994,1191,9,7,0,28,7,1,0,0
2,2022-07-18,25574.30,3651.24,3744.90,0.00,0.0,0.0,21157.85,54128.29,41290.56,95418.85,2086,1183,10,7,0,29,7,2,0,0
3,2022-07-25,25574.30,3651.24,53479.90,5730.00,0.0,0.0,35958.85,124394.29,68711.50,193105.79,2044,1020,5,7,0,30,7,3,0,0
4,2022-08-01,25574.30,3651.24,60952.90,2996.45,0.0,0.0,36106.01,129280.90,37677.68,166958.58,2113,1127,5,7,0,31,8,4,0,0


#### Check for duplicate weeks

In [20]:
data["week_start"].nunique()

156

#### Channel Consolidation

Combine digital with Social and OOH with Other to reduce the number of advertising channels, particularly those with limited spending observations.

In [21]:
# Combine 'digital' and 'social' spend
data["spend_safety_digital_social"] = data["spend_safety_digital"] + data["spend_safety_social"]

# Combine 'out of home' and 'other' spend
data["spend_safety_ooh_other"] = data["spend_safety_ooh"] + data["spend_safety_other"]

data.head(5)

,week_start,spend_safety_tv,spend_safety_radio,spend_safety_print,spend_safety_digital,spend_safety_social,spend_safety_ooh,spend_safety_other,spend_safety_total,spend_nonsafety_total,spend_total,crashes,injured,killed,days_observed,partial_week,iso_week,month,t,is_holiday_week,post_congestion,spend_safety_digital_social,spend_safety_ooh_other
0,2022-07-04,0.00,0.00,0.00,0.00,0.0,0.0,0.00,0.00,274.00,274.00,1919,1053,12,7,0,27,7,0,1,0,0.00,0.00
1,2022-07-11,21920.83,3129.63,3209.92,0.00,0.0,0.0,18135.30,46395.68,9356.00,55751.68,1994,1191,9,7,0,28,7,1,0,0,0.00,18135.30
2,2022-07-18,25574.30,3651.24,3744.90,0.00,0.0,0.0,21157.85,54128.29,41290.56,95418.85,2086,1183,10,7,0,29,7,2,0,0,0.00,21157.85
3,2022-07-25,25574.30,3651.24,53479.90,5730.00,0.0,0.0,35958.85,124394.29,68711.50,193105.79,2044,1020,5,7,0,30,7,3,0,0,5730.00,35958.85
4,2022-08-01,25574.30,3651.24,60952.90,2996.45,0.0,0.0,36106.01,129280.90,37677.68,166958.58,2113,1127,5,7,0,31,8,4,0,0,2996.45,36106.01


#### Drop number of rows where less than 7 days

In [22]:
data = data[data["days_observed"] == 7].copy()
data.shape

(156, 23)

#### Drop unneeded columns

In [23]:
data = data[[
    "week_start", 
    "spend_safety_tv", 
    "spend_safety_radio", 
    "spend_safety_print",
    "spend_safety_digital_social", 
    "spend_safety_ooh_other", 
    "crashes",
    "injured", 
    "killed",
    "t",
    "is_holiday_week",
    "post_congestion"
]]

#### Final check of data

In [24]:
# Data types and nulls
data_types_null_counts = pd.DataFrame()
data_types_null_counts["Data Type"] = data.dtypes
data_types_null_counts["Count of Nulls"] = data.isna().sum()
data_types_null_counts

,Data Type,Count of Nulls
week_start,datetime64[ns],0
spend_safety_tv,float64,0
spend_safety_radio,float64,0
spend_safety_print,float64,0
spend_safety_digital_social,float64,0
spend_safety_ooh_other,float64,0
crashes,int64,0
injured,int64,0
killed,int64,0
t,int64,0


#### Export data

In [25]:
data.to_parquet(
    CLEAN_DATA_DIR / "01_dot_mmm_preprocessing.parquet",
    index=False
)